# Init

In [0]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("gold.dim_customers")

# Transformation Logic

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY crm_c.customer_id) as customer_key,
    crm_c.customer_id,
    crm_c.customer_number,
    crm_c.first_name,
    crm_c.last_name,
    erp_loc.country,
    crm_c.marital_status,
    CASE 
        WHEN crm_c.gender <> 'Unknown' THEN crm_c.gender
        ELSE COALESCE(erp_c.gender, 'Unknown') 
    END as gender,
    erp_c.birth_date,
    crm_c.created_date
FROM silver.crm_customers crm_c
LEFT JOIN silver.erp_customers erp_c
    ON crm_c.customer_number = erp_c.customer_number
LEFT JOIN silver.erp_customer_location erp_loc
    ON crm_c.customer_number = erp_loc.customer_number
"""
df = spark.sql(query)

# Write into Gold Table

In [0]:
try:
    df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")
except Exception as e:
    logger.error(f"Failed to write Gold table: {e}")
    raise